# Vision Inference Benchmark

Compare MobileNetV3-Small ONNX FP32 and static INT8 inference on CPU.
Run the cells from top to bottom after setting the image paths below.

**Status:** the benchmark pipeline is implemented. The published sample run
changed its top prediction from **Arabian camel to pinwheel** after quantization.
Held-out accuracy is unmeasured; the INT8 model has not passed correctness validation.

The recorded run and metric definitions are in [README.md](README.md).
C++ inference and Unity/OpenXR integration are separate planned extensions.


## 1. Imports and configuration

Set the sample image and calibration directory. Calibration images are not bundled. New artifacts stay under `artifacts/`; the published root-level result snapshot is preserved.


In [ ]:
import json
from pathlib import Path
import platform
import subprocess
import sys
import tempfile
import time

import numpy as np
import onnxruntime as ort
from onnxruntime.quantization import (
    CalibrationDataReader,
    CalibrationMethod,
    QuantFormat,
    QuantType,
    quantize_static,
)
from onnxruntime.quantization.shape_inference import quant_pre_process
from PIL import Image
import torch
import torchvision
from torchvision.models import MobileNet_V3_Small_Weights, mobilenet_v3_small

sample_image_path = Path.home() / "Downloads" / "camel.jpg"
calibration_dir = Path("/tmp/onnx/calibration")
artifact_dir = Path.cwd() / "artifacts"

warmup_runs = 20
measured_runs = 200
intra_op_threads = 1
inter_op_threads = 1

# Optional: (held-out image path, ImageNet class index from 0 to 999).
# Imagenette's labels 0-9 must be mapped to ImageNet indices first.
eval_samples = []

if not sample_image_path.is_file():
    raise FileNotFoundError(f"Set sample_image_path to an existing image: {sample_image_path}")
calibration_image_paths = sorted(calibration_dir.glob("*.jpg"))
if not calibration_image_paths:
    raise ValueError(f"Add calibration JPG images or update calibration_dir: {calibration_dir}")
if warmup_runs < 0 or measured_runs <= 0:
    raise ValueError("warmup_runs must be non-negative and measured_runs must be positive")

artifact_dir.mkdir(parents=True, exist_ok=True)
fp32_path = artifact_dir / "mobilenet_fp32.onnx"
prepared_fp32_path = artifact_dir / "pre_quant_mobilenet_fp32.onnx"
int8_path = artifact_dir / "mobilenet_int8.onnx"

print(f"Calibration images: {len(calibration_image_paths)}")
print(f"Artifacts: {artifact_dir}")


## 2. Load the model and preprocess the image

Use the pretrained weights' resize/crop/normalization recipe. Evaluation mode controls training-specific layers; inference mode disables gradient bookkeeping.


In [ ]:
weights = MobileNet_V3_Small_Weights.IMAGENET1K_V1
model = mobilenet_v3_small(weights=weights)
model.eval()
preprocess = weights.transforms()
categories = weights.meta["categories"]

def load_image_array(path):
    with Image.open(path) as image:
        return preprocess(image.convert("RGB")).unsqueeze(0).numpy()

input_array = load_image_array(sample_image_path)
batch = torch.from_numpy(input_array)
assert input_array.shape == (1, 3, 224, 224)
assert input_array.dtype == np.float32

with torch.inference_mode():
    probabilities = model(batch).softmax(dim=1)[0]

print(f"Input: {input_array.shape}, {input_array.dtype}")
print("PyTorch top-5 predictions:")
for class_index in probabilities.topk(5).indices.tolist():
    print(f"  {categories[class_index]:20s} {probabilities[class_index].item():.4%}")


## 3. Export and prepare ONNX FP32

The example batch supplies the input shape and dtype for export. Graph preparation performs shape inference and optimization; it does not quantize weights.


In [ ]:
_ = torch.onnx.export(
    model,
    (batch,),
    fp32_path,
    dynamo=True,
    external_data=False,
    input_names=["input"],
    output_names=["logit"],
)
quant_pre_process(fp32_path, prepared_fp32_path)
print(f"Prepared FP32 model: {prepared_fp32_path.name}")


## 4. Calibrate and quantize to INT8

MinMax calibration observes activation ranges on the supplied images. QDQ represents quantization boundaries; supported runtime kernels determine how the graph executes. Per-channel quantization applies to supported weights.


In [ ]:
class ImageCalibrationDataReader(CalibrationDataReader):
    def __init__(self, image_paths, input_name):
        self.image_paths = iter(image_paths)
        self.input_name = input_name

    def get_next(self):
        path = next(self.image_paths, None)
        if path is None:
            return None
        return {self.input_name: load_image_array(path)}

# Read the graph's input name instead of hardcoding an exporter-generated name.
import onnx

prepared_model = onnx.load(prepared_fp32_path, load_external_data=False)
calibration_input_name = prepared_model.graph.input[0].name
del prepared_model

quantize_static(
    model_input=prepared_fp32_path,
    model_output=int8_path,
    calibration_data_reader=ImageCalibrationDataReader(
        calibration_image_paths, calibration_input_name
    ),
    quant_format=QuantFormat.QDQ,
    op_types_to_quantize=["Conv", "Gemm", "MatMul"],
    activation_type=QuantType.QInt8,
    weight_type=QuantType.QInt8,
    per_channel=True,
    calibrate_method=CalibrationMethod.MinMax,
)
print(f"Quantized model: {int8_path.name}")


## 5. Create CPU sessions and measure latency

Time only `session.run` on an already-preprocessed image. Session creation, model export, image loading, and preprocessing are excluded. Both sessions use matching CPU settings.


In [ ]:
options = ort.SessionOptions()
options.intra_op_num_threads = intra_op_threads
options.inter_op_num_threads = inter_op_threads
options.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL

fp32_session = ort.InferenceSession(
    str(fp32_path), sess_options=options, providers=["CPUExecutionProvider"]
)
int8_session = ort.InferenceSession(
    str(int8_path), sess_options=options, providers=["CPUExecutionProvider"]
)

def measure_latency(session):
    feed = {session.get_inputs()[0].name: input_array}
    for _ in range(warmup_runs):
        session.run(None, feed)

    durations_ms = []
    for _ in range(measured_runs):
        start = time.perf_counter_ns()
        session.run(None, feed)
        durations_ms.append((time.perf_counter_ns() - start) / 1_000_000)

    return {
        "median_ms": float(np.median(durations_ms)),
        "p95_ms": float(np.percentile(durations_ms, 95)),
    }

fp32_latency = measure_latency(fp32_session)
int8_latency = measure_latency(int8_session)
print("FP32:", fp32_latency)
print("INT8:", int8_latency)


## 6. Compare outputs, sizes, and benchmark conditions

Logits are raw class scores. Top-1 agreement compares predictions; it is not labeled accuracy. Non-finite outputs are rejected, and prediction disagreement is reported explicitly.


In [ ]:
fp32_output = fp32_session.run(
    None, {fp32_session.get_inputs()[0].name: input_array}
)[0]
int8_output = int8_session.run(
    None, {int8_session.get_inputs()[0].name: input_array}
)[0]

if not (np.isfinite(fp32_output).all() and np.isfinite(int8_output).all()):
    raise ValueError("Model outputs contain non-finite values")

difference = np.abs(fp32_output - int8_output)
cpu_model = platform.processor()
if platform.system() == "Darwin":
    cpu_model = subprocess.run(
        ["sysctl", "-n", "machdep.cpu.brand_string"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()

results = {
    "fp32": {**fp32_latency, "file_bytes": fp32_path.stat().st_size},
    "int8": {**int8_latency, "file_bytes": int8_path.stat().st_size},
    "median_speedup": fp32_latency["median_ms"] / int8_latency["median_ms"],
    "file_size_reduction_pct": 100 * (
        1 - int8_path.stat().st_size / fp32_path.stat().st_size
    ),
    "single_image_comparison": {
        "max_absolute_logit_difference": float(difference.max()),
        "mean_absolute_logit_difference": float(difference.mean()),
        "fp32_top1": int(fp32_output[0].argmax()),
        "int8_top1": int(int8_output[0].argmax()),
        "top1_agreement": bool(fp32_output[0].argmax() == int8_output[0].argmax()),
    },
    "conditions": {
        "cpu_model": cpu_model,
        "os": platform.platform(),
        "architecture": platform.machine(),
        "python": platform.python_version(),
        "torch": torch.__version__,
        "torchvision": torchvision.__version__,
        "onnxruntime": ort.__version__,
        "onnx": onnx.__version__,
        "weights": weights.name,
        "calibration_image_count": len(calibration_image_paths),
        "fp32_model_path": str(fp32_path),
        "int8_model_path": str(int8_path),
        "fp32_providers": fp32_session.get_providers(),
        "int8_providers": int8_session.get_providers(),
        "input_shape": list(input_array.shape),
        "input_dtype": str(input_array.dtype),
        "fp32_intra_op_threads": fp32_session.get_session_options().intra_op_num_threads,
        "int8_intra_op_threads": int8_session.get_session_options().intra_op_num_threads,
        "fp32_inter_op_threads": fp32_session.get_session_options().inter_op_num_threads,
        "int8_inter_op_threads": int8_session.get_session_options().inter_op_num_threads,
        "fp32_execution_mode": str(fp32_session.get_session_options().execution_mode),
        "int8_execution_mode": str(int8_session.get_session_options().execution_mode),
        "warmup": warmup_runs,
        "repeats": measured_runs,
        "timing_scope": "session.run on one preprocessed image; excludes loading and preprocessing",
        "thread_count_note": "0 means runtime default, not zero execution threads",
    },
    "accuracy": None,
    "memory": None,
}
print(json.dumps(results, indent=2))

print("FP32 top-1:", categories[results["single_image_comparison"]["fp32_top1"]])
print("INT8 top-1:", categories[results["single_image_comparison"]["int8_top1"]])
if not results["single_image_comparison"]["top1_agreement"]:
    print("VALIDATION WARNING: predictions differ; INT8 correctness is unresolved.")


## 7. Optional held-out accuracy

Populate `eval_samples` in the configuration cell, using separate images and their ImageNet class indices. An empty set records `accuracy: null`, not zero accuracy. The overlap check detects identical file paths, not renamed copies of the same image.


In [ ]:
def evaluate_accuracy(session, samples):
    if not samples:
        raise ValueError("Evaluation dataset is empty")

    input_name = session.get_inputs()[0].name
    calibration_paths = {path.resolve() for path in calibration_image_paths}
    correct = 0
    for image_path, imagenet_class_index in samples:
        if Path(image_path).resolve() in calibration_paths:
            raise ValueError("Use held-out images, not calibration images")
        if (
            not isinstance(imagenet_class_index, (int, np.integer))
            or not 0 <= imagenet_class_index < 1000
        ):
            raise ValueError("Labels must be ImageNet integer indices from 0 to 999")
        output = session.run(None, {input_name: load_image_array(image_path)})[0]
        correct += int(output[0].argmax() == imagenet_class_index)

    return {"count": len(samples), "top1_accuracy_pct": 100 * correct / len(samples)}

results["accuracy"] = None
if eval_samples:
    fp32_accuracy = evaluate_accuracy(fp32_session, eval_samples)
    int8_accuracy = evaluate_accuracy(int8_session, eval_samples)
    results["accuracy"] = {
        "fp32": fp32_accuracy,
        "int8": int8_accuracy,
        "accuracy_drop_percentage_points": (
            fp32_accuracy["top1_accuracy_pct"] - int8_accuracy["top1_accuracy_pct"]
        ),
    }
    print(json.dumps(results["accuracy"], indent=2))
else:
    print("Accuracy not measured: populate eval_samples with separate labeled images.")


## 8. Peak process memory

Run each model in a fresh Python process on macOS/Linux. Peak RSS includes Python, NumPy, ONNX Runtime, model loading, and inference; it is not weights-only or steady-state memory.


In [ ]:
memory_worker = r"""
import json
import platform
import resource
import sys
import numpy as np
import onnxruntime as ort

array = np.load(sys.argv[2], allow_pickle=False)
options = ort.SessionOptions()
options.intra_op_num_threads = int(sys.argv[3])
options.inter_op_num_threads = int(sys.argv[4])
options.execution_mode = getattr(ort.ExecutionMode, sys.argv[5])
session = ort.InferenceSession(
    sys.argv[1], sess_options=options, providers=["CPUExecutionProvider"]
)
feed = {session.get_inputs()[0].name: array}
for _ in range(int(sys.argv[6]) + int(sys.argv[7])):
    session.run(None, feed)

peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
# macOS reports bytes; Linux reports KiB.
peak_bytes = int(peak if platform.system() == "Darwin" else peak * 1024)
print(json.dumps({
    "peak_process_rss_bytes": peak_bytes,
    "peak_process_rss_mib": peak_bytes / 2**20,
}))
"""

if platform.system() not in {"Darwin", "Linux"}:
    raise RuntimeError("This RSS measurement supports macOS and Linux")

memory = {}
with tempfile.TemporaryDirectory() as temporary_directory:
    array_path = Path(temporary_directory) / "input.npy"
    np.save(array_path, input_array, allow_pickle=False)
    for name, model_path, session in [
        ("fp32", fp32_path, fp32_session),
        ("int8", int8_path, int8_session),
    ]:
        session_options = session.get_session_options()
        completed = subprocess.run(
            [sys.executable, "-c", memory_worker, str(model_path), str(array_path),
             str(session_options.intra_op_num_threads),
             str(session_options.inter_op_num_threads),
             session_options.execution_mode.name,
             str(warmup_runs), str(measured_runs)],
            capture_output=True, text=True,
        )
        if completed.returncode:
            raise RuntimeError(f"{name} memory measurement failed:\n{completed.stderr}")
        memory[name] = json.loads(completed.stdout)

results["memory"] = {
    "scope": "Peak whole-process RSS in a fresh process per model, including runtime overhead",
    **memory,
}
print(json.dumps(results["memory"], indent=2))


## 9. Save the run

Write the current results to `artifacts/benchmark_results.json`. Rerunning overwrites that generated file, but does not replace the published root-level snapshot. Run this cell again after repeating any measurements.


In [ ]:
results_path = artifact_dir / "benchmark_results.json"
results_path.write_text(json.dumps(results, indent=2, allow_nan=False) + "\n")
print(f"Saved: {results_path}")
